# 02: Text Preprocessing & Stratified Train/Val/Test Split
**Owner:** Poojan (Person 1 - `poojan_p1`)
**Tasks:** P1.10 - P1.13

### Objective:
Build a reproducible preprocessing function that cleans HTML, handles URLs/emails, and strictly preserves domain-specific technical tokens (C++, C#, SQL, Python, AWS, .NET, HTML/CSS). Split the data stratified into Train (70%), Val (15%), Test (15%).

### Design Decisions:
- **Stopword removal**: NOT applied in `clean_resume_text()` — left to P2 (TF-IDF has built-in stopword param) and P3 (Word2Vec may benefit differently)
- **Lemmatization**: NOT applied in `clean_resume_text()` — optional `postprocess_text()` helper provided
- **Technical tokens**: C++, C#, .NET, CI/CD, Node.js, Vue.js, React.js, HTML/CSS, ASP.NET are protected via regex substitution before punctuation removal

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.data_loader import load_resume_csv, stratified_split
from src.preprocessing import clean_resume_text, PROTECTED_TERMS

df = load_resume_csv()
print(f'Loaded {len(df)} records')
print(f'Protected terms: {list(PROTECTED_TERMS.values())}')

## P1.10: Apply Preprocessing & Inspect Results

In [ ]:
df['cleaned_text'] = df['Resume_str'].astype(str).apply(clean_resume_text)

print('Sample cleaned text preview:')
print('ORIGINAL:', str(df['Resume_str'].iloc[0])[:200])
print()
print('CLEANED:', df['cleaned_text'].iloc[0][:200])

## P1.11: Verify Technical Token Preservation

In [ ]:
test_cases = [
    ('C++', 'cplusplus', 'Experienced in C++ and data structures'),
    ('C#', 'csharp', 'Developed apps using C# and .NET'),
    ('.NET', 'dotnet', 'Built REST APIs with .NET 6'),
    ('CI/CD', 'cicd', 'Implemented CI/CD pipelines'),
    ('Node.js', 'nodejs', 'Backend with Node.js and Express'),
    ('HTML/CSS', 'htmlcss', 'Web design with HTML/CSS'),
    ('Python', 'python', 'ML with Python and TensorFlow'),
    ('SQL', 'sql', 'Database admin with SQL Server'),
    ('AWS', 'aws', 'Cloud certified in AWS'),
]

print(f'{"Term":<12} {"Expected":<12} {"Found?":<8} {"Cleaned Output"}')
print('-' * 70)
for term, expected, sample in test_cases:
    cleaned = clean_resume_text(sample)
    found = expected in cleaned
    print(f'{term:<12} {expected:<12} {"YES" if found else "NO":<8} {cleaned}')

## P1.12: Data Cleaning, Stratified Split & Export to data/processed/

In [ ]:
# Drop empty, very short, and duplicate records
empty_mask = df['Resume_str'].isna() | (df['Resume_str'].astype(str).str.strip() == '')
short_mask = df['Resume_str'].astype(str).apply(lambda x: len(x.split()) < 10)
df_clean = df[~empty_mask & ~short_mask].copy()
dup_mask = df_clean['cleaned_text'].duplicated(keep='first')
df_clean = df_clean[~dup_mask].copy()
print(f'Records after cleaning: {len(df_clean)} (dropped {len(df) - len(df_clean)})')

# Stratified split
train_df, val_df, test_df = stratified_split(df_clean, random_state=42)
print(f'Train: {len(train_df)} ({len(train_df)/len(df_clean)*100:.1f}%)')
print(f'Val:   {len(val_df)} ({len(val_df)/len(df_clean)*100:.1f}%)')
print(f'Test:  {len(test_df)} ({len(test_df)/len(df_clean)*100:.1f}%)')

# Export
os.makedirs('../data/processed', exist_ok=True)
cols = ['ID', 'Resume_str', 'cleaned_text', 'Category']
train_df[cols].to_csv('../data/processed/train.csv', index=False)
val_df[cols].to_csv('../data/processed/val.csv', index=False)
test_df[cols].to_csv('../data/processed/test.csv', index=False)
print('\nSplits saved to data/processed/')